# M1 实验队列 · Kaggle Runner

**用法**:Kaggle → New Notebook → File → Import Notebook 粘贴本文件(或从 GitHub `everest-an/M1` 的 `kaggle/kaggle_runner.ipynb` 导入)→ 右侧 Settings 选 **GPU T4** → Run All。

- 每个实验独立,跑完一个就把结果(`reasoning_depth.jsonl` + log)拷到 `/kaggle/working/`,**12h 会话被杀也不丢已完成的**。
- 会话结束后在 Output 面板下载 `m1_results.zip`,发回给 CC 入库。
- 断点续跑:把下面 `START` 改成上次完成的序号+1,重新 Run All。

**当前队列(2026-08-05)— parity k=32 长预算裁决 + J-Space J1 sweep**:
本地 6000 步 parity d32 全 chance(含 transformer 0.557)= 预算墙;拉到 12k 步验证 selective 能否 grok k=32。J1 是 workspace 驻留深度探针(队列躺 5 天)。纯 LNN 探针(`--attention_layers` 空)是有效协议(hybrid 被 attention 兜底,已证 null)。

In [ ]:
import subprocess, os, shutil, time, glob

# ── 环境 ──
if not os.path.exists('/kaggle/working/M1'):
    subprocess.run(['git','clone','--depth=1','https://github.com/everest-an/M1.git','/kaggle/working/M1'], check=True)
os.chdir('/kaggle/working/M1')
subprocess.run(['git','pull','--ff-only'], check=False)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# ── 实验队列(独立可重排;想换实验只改这里)──
PARITY = ('python benchmarks/reasoning_depth.py --task parity --difficulty 32 '
          '--mix --eval_depths 1 --attention_layers ')
QUEUE = [
    # A. parity k=32 selective 长预算(纯 LNN,12k 步 — 本地 6000 步全 chance=预算墙)
    PARITY + '--seeds 0 1 2 --steps 12000 --selective_decay --skip_transformer --tag kg-parity-d32-sel',
    # B. parity k=32 stock 同预算对照(预言: 仍卡 chance,确认预算墙非机制)
    PARITY + '--seeds 0 1 2 --steps 12000 --tag kg-parity-d32-stock',
    # C. J-Space J1 工作区驻留 sweep(pointer_chase 课程任务,workspace_iterations 深度)
    'python benchmarks/reasoning_depth.py --task pointer_chase --n_values 8 --mode fixed '
    '--mix --difficulty 4 --steps 10000 --seeds 0 1 2 --eval_depths 1 2 4 --workspace --tag kg-j1',
]
START = 0  # 断点续跑:改成上次完成的序号+1

def snapshot():
    for f in glob.glob('benchmarks/results/*.jsonl') + glob.glob('benchmarks/results/*.log'):
        shutil.copy(f, '/kaggle/working/')

for i, cmd in enumerate(QUEUE):
    if i < START: continue
    print(f'\n══════ [{i}] {cmd}\n', flush=True)
    t0 = time.time()
    r = subprocess.run(cmd.split(), capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print('STDERR:', r.stderr[-2000:])
    print(f'[{i}] 用时 {(time.time()-t0)/60:.1f} min, exit={r.returncode}', flush=True)
    snapshot()
print('\n全部完成')

In [ ]:
# ── 打包下载 ──
import shutil
shutil.make_archive('/kaggle/working/m1_results', 'zip', '/kaggle/working/M1/benchmarks/results')
print('下载 /kaggle/working/m1_results.zip,发回给 CC 入库')